# Phase 4b: Improved ML Classification

Three improvements to boost LOSO-CV results:

1. **Feature Selection**: 
    - Keep only the most informative features
    - Reduces noise from weakly-predictive features
    - *Method:* SelectKBest with ANOVA F-test

2. **Class Balancing:**
    - Handle the minority "High Stress" class
    - SVM and RF get class_weight = 'balanced'
    - Makes the model pay more attention to rare classes

3. **Hyperparameter Tuning:**
    - Find the best settings for each classifier
    - Use a small grid search inside LOSO-CV
    - *Method:* Try a few sensible values and pick what works best

Each improvements addresses a different problem:
- Feature selection fights the curse of dimensionality
- Class balancing fights label imbalance
- Hyperparameter tuning fights model misconfiguration

**Prerequisites:**
- Phase 3 completed (features.npz exists in Results/phase3/)

In [ ]:
# Imports:

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)
import time
import warnings
warnings.filterwarnings('ignore')
 
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

# Configuration

In [ ]:
BASE_DIR = os.path.join("..", "")
 
DATA_DIR = os.path.join(BASE_DIR, "Results", "phase3")
OUTPUT_DIR = os.path.join(BASE_DIR, "Results", "phase4b")
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
# Number of top features to keep
# 50-100 is typical for EEG stress tasks with ~2000-3000 samples
TOP_K_FEATURES = 80

# Improved LOSO-CV with Feature Selection & Class Weighting

Enhanced LOSO-CV with:
1. Feature selection (ANOVA F-test, top-K features)
2. Feature scaling (StandardScaler)
3. Class balancing (via class_weight in classifier)

Feature selection is done inside the LOSO loop, using only training data. Doing it on all data first would leak info from the test subject to the training phase.

In [3]:

"""
Parameters:
    X: feature matrix (n_samples, n_features)
    y: labels (n_samples,)
    subjects: subject IDs (n_samples,)
    classifier_fn: function returning fresh classifier
    classifier_name: for printing
    k_features: how many top features to keep
    use_class_weight: whether to apply class balancing

Returns:
    results dict (same format as Phase 4)
"""

def loso_cv_improved(X, y, subjects, classifier_fn, classifier_name="Classifier", k_features=TOP_K_FEATURES, use_class_weight=True):
    
    unique_subjects = sorted(np.unique(subjects))
    n_subjects = len(unique_subjects)
    
    all_true = []
    all_pred = []
    subject_accuracies = {}
    selected_features_counts = np.zeros(X.shape[1])  # Track which features get picked
    
    print(f"\n  Running LOSO-CV with {classifier_name}...")
    print(f"    Feature selection: top {k_features} features (ANOVA F-test)")
    print(f"    Class weighting: {'enabled' if use_class_weight else 'disabled'}")
    
    start_time = time.time()
    
    for i, test_subject in enumerate(unique_subjects):
        test_mask = subjects == test_subject
        train_mask = ~test_mask
        
        X_train, X_test = X[train_mask], X[test_mask]
        y_train, y_test = y[train_mask], y[test_mask]
        
        # Step 1: Feature selection using training data only
        selector = SelectKBest(f_classif, k=min(k_features, X_train.shape[1]))
        X_train_sel = selector.fit_transform(X_train, y_train)
        X_test_sel = selector.transform(X_test)
        
        # Track which features got selected
        selected_mask = selector.get_support()
        selected_features_counts[selected_mask] += 1
        
        # Step 2: Scale features (fit on train only)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_sel)
        X_test_scaled = scaler.transform(X_test_sel)
        
        # Step 3: Train with class weights if enabled
        clf = classifier_fn()
        if use_class_weight and hasattr(clf, 'class_weight'):
            clf.set_params(class_weight='balanced')
        
        clf.fit(X_train_scaled, y_train)
        y_pred = clf.predict(X_test_scaled)
        
        all_true.extend(y_test)
        all_pred.extend(y_pred)
        
        acc = accuracy_score(y_test, y_pred)
        subject_accuracies[test_subject] = acc
        
        if (i + 1) % 10 == 0 or i == 0:
            elapsed = time.time() - start_time
            remaining = elapsed / (i + 1) * (n_subjects - i - 1)
            print(f"    Subject {i+1}/{n_subjects} "
                  f"(acc: {acc:.1%}) "
                  f"[{elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining]")
    
    total_time = time.time() - start_time
    all_true = np.array(all_true)
    all_pred = np.array(all_pred)
    
    overall_acc = accuracy_score(all_true, all_pred)
    unique_labels = np.unique(all_true)
    avg_method = 'binary' if len(unique_labels) == 2 else 'weighted'
    pos_label = unique_labels[1] if len(unique_labels) == 2 else None
    
    precision = precision_score(all_true, all_pred, average=avg_method, pos_label=pos_label, zero_division=0)
    recall = recall_score(all_true, all_pred, average=avg_method, pos_label=pos_label, zero_division=0)
    f1 = f1_score(all_true, all_pred, average=avg_method, pos_label=pos_label, zero_division=0)
    cm = confusion_matrix(all_true, all_pred)
    
    results = {
        'classifier': classifier_name,
        'accuracy': overall_acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm,
        'all_true': all_true,
        'all_pred': all_pred,
        'subject_accuracies': subject_accuracies,
        'time': total_time,
        'unique_labels': unique_labels,
        'selected_features_counts': selected_features_counts,
    }
    
    print(f"    Done in {total_time:.1f}s — Overall accuracy: {overall_acc:.1%}")
    
    return results